<a href="https://colab.research.google.com/github/RagghavAmola/Jet-Engine-RUL-Prediction/blob/main/RUL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ---- IMPORTS ----
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Masking, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# ---- DATA LOADING ----
def load_file(path):
    cols = ["engine","cycle"]+[f"op{i}" for i in range(1,4)]+[f"s{i}" for i in range(1,22)]
    return pd.read_csv(path, sep="\s+", names=cols, header=None)

# Using simple file names, assuming files are uploaded to Colab runtime
train = load_file("train_FD001.txt")
test = load_file("test_FD001.txt")
true_rul = pd.read_csv("RUL_FD001.txt", header=None)

# ---- CALCULATE RUL ----
train["RUL"] = train.groupby("engine")["cycle"].transform(lambda x: x.max() - x)
# ENHANCEMENT: RUL Clipping (Crucial for CMAPSS)
train["RUL"] = train["RUL"].clip(upper=125)

# ---- Quick EDA ----
plt.figure(figsize=(8,3))
train["RUL"].hist(bins=40)
plt.title("Clipped RUL Distribution in Training Data")
plt.xlabel("Remaining Useful Life (Clipped at 125)")
plt.ylabel("Frequency")
plt.show()

# ---- FEATURE SELECTION ----
remove = ["engine","cycle","RUL"]
features = [c for c in train.columns if c not in remove]

# ---- SCALING ----
scaler = StandardScaler()
train[features] = scaler.fit_transform(train[features])
test[features] = scaler.transform(test[features])

# ---- WINDOWING FUNCTION ----
def make_windows(df, window=50):
    X,y = [],[]
    for e in df["engine"].unique():
        temp = df[df["engine"]==e]
        vals = temp[features].values
        rulv = temp["RUL"].values
        for i in range(len(temp)-window):
            X.append(vals[i:i+window])
            y.append(rulv[i+window])
    return np.array(X), np.array(y)

X_train, y_train = make_windows(train, window=50)

def make_test(df, true_rul, window=50):
    X,y = [],[]
    for i,e in enumerate(df["engine"].unique()):
        temp = df[df["engine"]==e][features].values
        pad = window-len(temp)

        # DEBUG FIX: Padding with zeros for the Masking layer
        if pad>0:
            temp = np.vstack([np.zeros((pad, len(features))), temp])

        X.append(temp[-window:])
        y.append(true_rul.iloc[i,0])

    return np.array(X), np.array(y)

X_test, y_test = make_test(test, true_rul)

# ---- MODEL ----
model = Sequential([
    Masking(mask_value=0.0, input_shape=(50,len(features))),
    Bidirectional(LSTM(64, return_sequences=True), name='lstm_feature_map'),
    Dropout(0.2),
    Bidirectional(LSTM(32)),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1)
])

model.compile(optimizer="adam", loss="mse", metrics=[tf.keras.metrics.RootMeanSquaredError()])
model.summary()

# ---- TRAIN MODEL ----
stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=25,
    batch_size=64,
    callbacks=[stop],
    verbose=1
)

# ---- TRAINING CURVE ----
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.legend(); plt.title("Training Performance"); plt.show()

# ---- EVALUATION ----
pred = model.predict(X_test).flatten()
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("\n✅ Test RMSE:", rmse)
print("✅ Test R²:", r2)

# ---- TRUE vs PREDICT ----
plt.scatter(y_test, pred)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], "r--")
plt.xlabel("True RUL"); plt.ylabel("Predicted RUL")
plt.title("RUL Prediction Accuracy"); plt.show()

# ---- PLOT EACH ENGINE ----
plt.figure(figsize=(12, 5))
plt.plot(y_test, label="True")
plt.plot(pred, label="Predicted")
plt.legend(); plt.title("Engine-wise RUL"); plt.show()

# ---- SIMPLE FEATURE IMPORTANCE (Mean Correlation to RUL) ----
corr = train.corr()["RUL"].drop(["cycle","engine","RUL"])
corr.abs().sort_values().plot(kind="bar", figsize=(10,4))
plt.title("Feature Importance (Correlation Based)")
plt.show()

# ---- Deployment Placeholder ----
model.save("rul_model.h5")
import pickle
pickle.dump(scaler, open("scaler.pkl","wb"))
print("\n📦 Model & scaler saved. Ready for basic deployment!\n")

# ======================= GRAD-CAM =======================

def compute_grad_cam(model, input_data, target_layer_name):
    """
    Computes the Grad-CAM activation map for a time-series input.
    Uses an imperative approach (manual layer application) to avoid Keras internal KeyErrors.
    """
    # 1. Define a model that outputs the target feature map
    feature_map_model = Model(
        model.inputs,
        model.get_layer(target_layer_name).output
    )

    # Find the index of the target layer
    target_layer_index = -1
    for i, layer in enumerate(model.layers):
        if layer.name == target_layer_name:
            target_layer_index = i
            break

    # 2. Use GradientTape to compute gradients
    with tf.GradientTape() as tape:
        # 2a. Calculate feature maps by running the input through the feature map model
        feature_maps = feature_map_model(input_data)
        tape.watch(feature_maps)

        # 2b. Manually apply the subsequent layers to calculate the final prediction
        x = feature_maps
        for i in range(target_layer_index + 1, len(model.layers)):
            x = model.layers[i](x)
        preds = x

        # We take the gradient of the scalar prediction w.r.t the feature map
        loss = preds[0][0] # The final scalar RUL prediction

    # 3. Compute the gradient of the prediction (loss) w.r.t the feature map
    grads = tape.gradient(loss, feature_maps)

    # 4. Global Average Pooling over the time dimension (axis=1) of the gradients
    pooled_grads = tf.reduce_mean(grads, axis=1)

    # 5. Weight the feature map with the importance of each channel (alpha * feature_map)
    # The output of the LSTM is 3D (Batch, Timesteps, Features), pooled_grads is 2D (Batch, Features)
    weighted_feature_map = feature_maps * pooled_grads[:, tf.newaxis, :]

    # 6. Sum across the feature channels (axis=2) to get the time-step importance (Activation Map)
    heatmap = tf.reduce_sum(weighted_feature_map, axis=-1)[0]

    # 7. Apply ReLU (sets negative contributions to zero)
    heatmap = np.maximum(heatmap.numpy(), 0)

    # 8. Normalize the heatmap
    heatmap /= (np.max(heatmap) + 1e-8)

    return heatmap

print("\n\n🔬 Running Grad-CAM for Time-Series Interpretation...")

# Select Engine #1 from the test set for visualization
engine_index = 0
single_engine_input = X_test[engine_index:engine_index+1]

# Compute the Grad-CAM heatmap (activation map)
target_layer_name = 'lstm_feature_map'
activation_map = compute_grad_cam(model, single_engine_input, target_layer_name)

# Get the RUL prediction for context
predicted_rul = model.predict(single_engine_input).flatten()[0]
true_rul_val = y_test[engine_index]


# ---- PLOT GRAD-CAM RESULTS ----
plt.figure(figsize=(10, 5))
cycles = np.arange(50) # The 50 cycles in the window
plt.bar(cycles, activation_map, color=plt.cm.jet(activation_map), width=1.0)
plt.title(f"Grad-CAM Time-Step Activation Map (Engine {engine_index+1})")
plt.xlabel(f"Cycle within Window (Cycle 49 is the most recent)")
plt.ylabel("Normalized Activation (Importance)")
plt.text(0.95, 0.95,
         f"Predicted RUL: {predicted_rul:.1f}\nTrue RUL: {true_rul_val}",
         transform=plt.gca().transAxes,
         fontsize=10, verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle="round,pad=0.5", fc="white", alpha=0.8))
plt.grid(axis='y', alpha=0.5)
plt.show()

print(f"\n✅ Grad-CAM activation map generated for Engine {engine_index+1}.")
print("The plot shows which of the last 50 cycles contributed most to the RUL prediction.")

In [ ]:
!pip install streamlit -q

In [ ]:
!pip install pyngrok streamlit


In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token(<AUTH_TOKEN>)


In [ ]:
import os
print(os.listdir('.'))

Alternatively, you can use the `!ls` shell command directly in a Colab cell to list the files in the current directory:

In [ ]:
!ls

In [ ]:
# Kill old tunnels
ngrok.kill()

# Create a public URL
public_url = ngrok.connect(8501)
public_url


In [ ]:
!streamlit run dashboard.py --server.port 8501 --server.address 0.0.0.0
